In [ ]:
!ls ../bert/albert/CLUEdataset/csl/train.json

In [ ]:
!head ../bert/albert/CLUEdataset/iflytek/train.json

In [ ]:
from fastai.vision.all import *
from torch.nn import *


def Flat(a):
    return np.array(list(a)).ravel().tolist()


def Zip(a, b):
    return np.stack((a, b), 1).ravel().tolist()


def Unzip(a):
    return np.array(a, "O").reshape(-1, 2).T


class Dat:
    m = {}

    def __init__(s, a, bs=4, train=1):
        a = [loads(i) for i in open(f"../bert/albert/CLUEdataset/iflytek/{a}.json")]
        if train:
            s.x = [torch.tensor([s.m.setdefault(a, len(s.m)) for a in a["sentence"]], device="cuda") for a in a]
        else:
            s.x = [torch.tensor([s.m[a] for a in a["sentence"] if a in s.m], device="cuda") for a in a]
        s.y = torch.tensor([int(a["label"]) for a in a], device="cuda")
        s.bs = bs

    def __len__(s):
        return len(s.x) // s.b

    def __iter__(s):
        for i in range(len(s)):
            i = torch.randint(len(s.x), [s.b], device="cuda")
            yield [torch.cat([s.x[i] for i in i]), np.array([len(s.x[i]) for i in i])], s.y[i]


class Bi(Sequential):
    def forward(s, x):
        return super().forward(x[0]), x[1]


class Red(Sequential):
    def forward(s, x):
        x, l = x
        p = np.array([x[:0]] * len(l), "O")
        x, r = Unzip(x.split(Zip(l // 2 * 2, l % 2)))
        while any(map(len, x)):
            x, p = Unzip(
                super()
                .forward(torch.cat(Zip(x, p)).view(-1, x[0].size(1) * 2))
                .split(Flat([t // 2 * 2, 1] if t % 2 and not len(r) else [t, 0] for x, p, r in zip(x, p, r) for t in [(len(x) + len(p)) // 2]))
            )
            for i in range(len(x)):
                if len(x[i]) % 2 or len(p[i]):
                    p[i], r[i] = r[i], p[i]
        return torch.cat(r.tolist())


def Rand(a):
    return (np.random.rand(len(a)) * (a + 0.999)).astype(int)


def Slice(a, i, l):
    return torch.stack(a.split(np.r_[i[0], np.pad(np.diff(i, append=len(a))[:, None] - l, ((0, 0), (1, 0)), constant_values=l).ravel()].tolist())[1::2])


class Dat2:
    m = {}

    def __init__(s, a, b=16, t=1):
        a = [loads(i) for i in open(a)]
        s.y = torch.tensor([int(a["label"]) for a in a], device="cuda")
        a = [[s.m.setdefault(a, len(s.m)) for a in a["sentence"]] if t else [s.m[a] for a in a["sentence"] if a in s.m] for a in a]
        s.x = torch.tensor([a for a in a for a in a], device="cuda")
        s.l = np.array([len(a) for a in a])
        s.L = np.r_[0, s.l[:-1]].cumsum()
        s.b = b

    def __len__(s):
        return len(s.y) // s.b

    def __iter__(s):
        for i in range(len(s)):
            i = np.random.choice(len(s.y), s.b, 0)
            i.sort()
            L = s.l[i]
            l = L.min()
            yield Slice(s.x, s.L[i] + Rand(L - l), l), s.y[torch.tensor(i, device="cuda")]


class Red2(Sequential):
    def forward(s, x):
        r = None
        while x.size(1) > 1 or r is not None:
            if x.size(1) % 2:
                if r is not None:
                    x = torch.cat((x, r), 1)
                    r = None
                else:
                    x, r = x.split(x.size(1) - 1, 1)
            x = super().forward(x.view(len(x), -1, x.size(2) * 2))
        return x[:, 0]

In [ ]:
m = Learner(
    DataLoaders(Dat("train"), Dat("dev", 2599, 0)),
    Sequential(Bi(Embedding(len(Dat.m), 128)), Red(Linear(256, 128), CELU(inplace=1)), Linear(128, 119)).cuda(),
    CrossEntropyLoss(),
    metrics=accuracy,
    cbs=ShowGraphCallback,
)
m.lr_find()

In [ ]:
m.fit_one_cycle(50, .001, 9, 99, .1)